In [1]:
%pip install -q "unstructured[md]" nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    separators=['\n\n', '\n']
)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%pip install -q "langchain>=1.0" "langchain-core>=1.4.8" "langchain-text-splitters>=1.0" langchain-community langchain-chroma langchain-google-genai sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_community.document_loaders import TextLoader

text_path = './documents/income_tax.txt'
loader = TextLoader(text_path)
document_list = loader.load_and_split(text_splitter)


/var/folders/8l/ylr41sxs3_l9cwqgdmznm12h0000gn/T/ipykernel_25468/2617797831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
document_list[47]

Document(metadata={'source': './documents/income_tax.txt'}, page_content='② 제70조제1항, 제70조의2에 따른 제74조에 따라 차례로 할 것이 제70조제1항제2호에 따르며 서류를 제출하여야 한다는 경우에는 기준소득 중 거주자 본인이 된다(분산)과 제70조제2와 제74조에 따른 제료 및 제대법을 포함한다. 단, 차별제표청정인 그 업체를 남겨 제출한 경우로 그에 대하여 아니하다.<개정 2013. 1. 1.>\n  ③ 제80조에 따른 수익과 관련의 경우에는 기초공제 중 거주자 본인이 된다(분산)과 그에 관한 적지사항을 분명히 한다.\n[전문개정 2009. 12. 31.]\n[제목개정 2014. 1. 1.]\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의2에 따른 세액감면을 적용하는 경우 제54조제3항에 따라 공동사업자의 소득에 합산과세되는 특별세액거래의 지출․납입․투자 등의 금액이 있을 경우 주된 공동사업자의 소득에 합산과세되는 소득금액에 합산되어 주된 공동사업자의 합산과세세액은 공동사업소득액 또는 공동사업창출세액을 계산할 때 소득공제 또는 세액공제를 받을 수 있다. \n[개정 2014. 1. 1.]\n[전문개정 2009. 12. 31.]\n[제목개정 2014. 1. 1.]\n제2절 세액의 계산 <개정 2009. 12. 31.>\n제1관 세율 <개정 2009. 12. 31.>\n제55조(세율) 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 "종합소득과세표준세액"이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n종합소득\n┌───────────────┐\n│ 과세표준의 6개 구간 │\n├───────────────┤\n│ 1,400만원 이하       

In [6]:
# Gemini 임베딩은 무료 할당량(quota) 초과 시 429 RESOURCE_EXHAUSTED가 발생할 수 있습니다.
# 로컬 HuggingFace 모델을 사용하면 API 할당량 없이 학습을 이어갈 수 있습니다.
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask"
)

# Gemini API 할당량이 충분할 때는 아래를 사용하세요.
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8890.31it/s]


In [7]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name="income_tax_collection",
    persist_directory="./income_tax_collection_hf"
)

In [8]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [9]:
query = '연봉 5천만원의 직장인의 소득세는?'

In [10]:
retriever.invoke(query)

[Document(id='3f985890-603d-4abe-ac58-b1356543feba', metadata={'source': './documents/income_tax.txt'}, page_content='소득세법\n\n「법인세법」 제67조에 따라 기타소득으로 처분된 소득  \n제20조제3항제1호나목 및 다목의 금액을 고소득의 성격으로 불구하고 연금에 수령한 소득  \n즉 주식 및 이러한 주식매입권을 퇴직 후에 행사하지 않거나 고용관계 없이 주식매입권을 부여받아 이행사항으로 인해  \n\n\n중임원 혹은 해당 직원의 퇴직한 후에 지급받는 직무발명상금  \n외람  \n알선수재 및 배임수재에 의해 받는 금품  \n사체 2020. 12. 29.  \n종교단체의 사자가 종교의식을 진행하거나 종교관련자료서적의 활동과 관련하여 대통령명으로 정하는 종교 단체로부터 받은 소득(이하 “종교소득”이라 한다)\n   ① 제1항 및 제19조제1항제21호에 불구하고 대통령명으로 정하는 서화(書畫)·공동체의 양도로 발생하는 소득(사업행위를 가진 등 대통령명으로 정한 경우에 발생하는 소득을 제외한다)는 기타소득으로 한다.  \n신설 2020. 12. 29.\n   ② 기타소득금액은 해당 과세기간의 총수입금액에서 이용된 필요경비를 공제한 금액으로 한다.  \n개정 2020. 12. 29.\n   ③ 제1제조26조에 따른 종교소득에 대하여 제20조제1항에 따른 근로소득으로 원천징수하고 이를 한 경우에는 해당 소득을 근로소득으로 본다.  \n<신설 2015. 12. 15. 2020. 12. 29.>\n   ④ 기타소득의 계정 법의 적용방법과 밖에 필요한 사항은 대통령으로 정한다.  \n개정 2015. 12. 15., 2020. 12. 29.  \n[문서변경 2009. 12. 31.]\n\n제2조(기타소득)\n기타소득은 이자소득· 배당소득· 사업소득· 근로소득· 영업소득· 금융투자소득 및 양도소득 외의 소득으로서 다음 각 호에 정규하는 것으로 한다.\n- <개정 2009. 7. 31., 

In [11]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class AgentState(TypedDict):
    query: str
    context: List[Document]
    answer: str

In [12]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(AgentState)

In [13]:
def retrieve(state: AgentState) -> AgentState:
    query = state['query']
    docs = retriever.invoke(query)
    return {'context': docs}

In [14]:
%pip install -q langsmith

Note: you may need to restart the kernel to use updated packages.


In [15]:
from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI

client = Client()
prompt = client.pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,
)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [16]:
def generate(state: AgentState) -> AgentState:
    context = "\n\n".join(doc.page_content for doc in state["context"])
    question = state["query"]
    rag_chain = prompt | llm
    response = rag_chain.invoke({"question": question, "context": context})
    return {"answer": response.content}

In [17]:
graph_builder.add_node('retrieve', retrieve)
graph_builder.add_node('generate', generate)

In [18]:
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'generate')
graph_builder.add_edge('generate', END)



In [19]:
graph = graph_builder.compile()

In [20]:
initial_state = {'query': query}
graph.invoke(initial_state)

{'query': '연봉 5천만원의 직장인의 소득세는?',
 'context': [Document(id='3f985890-603d-4abe-ac58-b1356543feba', metadata={'source': './documents/income_tax.txt'}, page_content='소득세법\n\n「법인세법」 제67조에 따라 기타소득으로 처분된 소득  \n제20조제3항제1호나목 및 다목의 금액을 고소득의 성격으로 불구하고 연금에 수령한 소득  \n즉 주식 및 이러한 주식매입권을 퇴직 후에 행사하지 않거나 고용관계 없이 주식매입권을 부여받아 이행사항으로 인해  \n\n\n중임원 혹은 해당 직원의 퇴직한 후에 지급받는 직무발명상금  \n외람  \n알선수재 및 배임수재에 의해 받는 금품  \n사체 2020. 12. 29.  \n종교단체의 사자가 종교의식을 진행하거나 종교관련자료서적의 활동과 관련하여 대통령명으로 정하는 종교 단체로부터 받은 소득(이하 “종교소득”이라 한다)\n   ① 제1항 및 제19조제1항제21호에 불구하고 대통령명으로 정하는 서화(書畫)·공동체의 양도로 발생하는 소득(사업행위를 가진 등 대통령명으로 정한 경우에 발생하는 소득을 제외한다)는 기타소득으로 한다.  \n신설 2020. 12. 29.\n   ② 기타소득금액은 해당 과세기간의 총수입금액에서 이용된 필요경비를 공제한 금액으로 한다.  \n개정 2020. 12. 29.\n   ③ 제1제조26조에 따른 종교소득에 대하여 제20조제1항에 따른 근로소득으로 원천징수하고 이를 한 경우에는 해당 소득을 근로소득으로 본다.  \n<신설 2015. 12. 15. 2020. 12. 29.>\n   ④ 기타소득의 계정 법의 적용방법과 밖에 필요한 사항은 대통령으로 정한다.  \n개정 2015. 12. 15., 2020. 12. 29.  \n[문서변경 2009. 12. 31.]\n\n제2조(기타소득)\n기타소득은 이자소득· 배당소득· 사업소득· 근로소득· 영업소득· 금융투자소득 및 양도소득 외의 소득으